# Automated LFIA Signal Quantification Pipeline
Welcome to the interactive Google Colab environment for lateral flow immunoassay analysis. This notebook allows you to upload assay images, run the YOLOv8 localization pipeline, and extract quantitative analyte concentrations automatically without installing anything locally.

In [ ]:
#@title 1. Install Dependencies & Setup Environment
#@markdown Click the play button to clone the repository and install all required libraries.

import os

print("⏳ Cloning repository and setting up environment...")
!git clone https://github.com/harishs-io/lateral_flowimmunoassay_quant.git
%cd lateral_flowimmunoassay_quant

# Install dependencies
!pip install -q ultralytics opencv-python matplotlib scipy pandas numpy

print("\n✅ Pipeline environment successfully loaded!")

In [ ]:
#@title 2. Upload Assay Image
#@markdown Run this cell to upload the LFIA image you want to analyze directly from your computer.

from google.colab import files

print("Please upload your assay image (e.g., .jpg, .png):")
uploaded = files.upload()
image_path = list(uploaded.keys())[0]

print(f"\n✅ Successfully uploaded: {image_path}")

In [ ]:
#@title 3. Run Pipeline & Extract LSPR Signal
#@markdown Adjust the detection parameters below if necessary, then run the cell:

yolo_confidence = 0.75 #@param {type:"slider", min:0.1, max:1.0, step:0.05}
baseline_correction = "Dynamic" #@param ["Dynamic", "Static", "None"]

# Import the modularized code from the repository
import sys
sys.path.append('.')
from src.lfia_analyzer import LFIAAnalyzer

print("⏳ Running LFIA Quantification Pipeline...")

# Initialize and run (Using try/except to handle paths if the user hasn't added weights yet)
try:
    analyzer = LFIAAnalyzer(weights='weights/best.pt', conf=yolo_confidence)
    results = analyzer.process_image(image_path, baseline=baseline_correction)
    
    print("✅ Analysis complete! Plotting chromatogram...")
    results.plot_chromatogram()
except Exception as e:
    print(f"\n⚠️ Pipeline Execution Error: {e}\nEnsure your model weights are uploaded to the 'weights/' folder.")

In [ ]:
#@title 4. View Calibration Results & Download Report
#@markdown This will display the 4PL regression and download the final quantification statistics.

from google.colab import files

try:
    results.show_calibration_curve()
    results.export_csv('quantification_report.csv')
    
    print("✅ Report generated. Downloading 'quantification_report.csv'...")
    files.download('quantification_report.csv')
except NameError:
    print("⚠️ Error: Run the pipeline in Step 3 first before exporting.")